In [ ]:
# run the TSNE of the test dataset

from tqdm import tqdm
import yaml, torch
import numpy as np
from sklearn.manifold import TSNE
from matplotlib import pyplot as plt
from torchvision.datasets import CIFAR10

from transforms import common_transform
from lejepa import init_model

run_name = 'run01' # Change this to the name of the run you want to load
device   = 'cpu'   # Change this to 'cuda' if you want to use GPU

In [ ]:
# Load the training parameters
with open(f'checkpoints/{run_name}/config.yaml', 'r') as f:
    params = yaml.safe_load(f)


# Initialize the model and load the checkpoint
encoder = init_model(params)
encoder = encoder.to(device)

checkpoint = torch.load(f'checkpoints/{run_name}/best_epoch.pth.tar', map_location=device)
encoder.load_state_dict(checkpoint['encoder'])

encoder.eval();


# Calculate the embeddings for the test set 
def get_embeddings(train):
    batch_size = 64

    dataset = CIFAR10(root='./data', train=train, transform=common_transform, download=True)
    dataloaders = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)

    # Calculate the embeddings
    embeddings = []
    labels = []
    with torch.no_grad():
        for data in tqdm(dataloaders):
            views, batch_labels = data
            emb = encoder(views.to(device))

            embeddings.extend(emb.cpu().numpy())
            labels.extend(batch_labels.numpy())
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    return embeddings, labels

embeddings_test, labels_test = get_embeddings(train=False)


# Run and plot the TSNE
tsne = TSNE(n_components=2, random_state=0)
embeddings_2d = tsne.fit_transform(embeddings_test)

plt.figure(figsize=(10, 6))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels_test, cmap='tab10', s=7, alpha=0.5)
plt.colorbar();

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

embeddings_train, labels_train = get_embeddings(train=True)

#classifier = LogisticRegression(max_iter=1000, random_state=0)
#classifier.fit(X_train, y_train)

classifier = RandomForestClassifier(n_estimators=100, random_state=0)
classifier.fit(embeddings_train, labels_train)

#classifier = KNeighborsClassifier(n_neighbors=20)
#classifier.fit(embeddings_train, labels_train)

labels_pred = classifier.predict(embeddings_test)

# Plot the confusion matrix
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
cm = confusion_matrix(labels_test, labels_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45, values_format='.2f', text_kw={'fontsize': 8})
plt.title('Confusion Matrix (Normalized)')
plt.show()

test_accuracy = (labels_pred == labels_test).sum().item() / labels_test.shape[0] 
print(f'Test Accuracy: {test_accuracy*100:.2f}%')